# Relay Simulation Results

This notebook loads `relay_comparison.pkl` generated by `relay_simlation.py` and plots scenario comparisons.

Run it using: cd /workspace && python3 relay_simlation.py

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path('relay_comparison.pkl'), Path('../relay_comparison.pkl')]
pkl_path = next((p for p in candidates if p.exists()), None)
if pkl_path is None:
    raise FileNotFoundError('Could not find relay_comparison.pkl in current or parent directory.')

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

print(f'Loaded: {pkl_path.resolve()}')
print('Keys:', list(data.keys()))

In [ ]:
n_nodes = np.array(data['nNodes'])
scenarios = data['scenarios']

summary = []
for name, vals in scenarios.items():
    summary.append({
        'scenario': name,
        'points_success': len(vals.get('success', [])),
        'points_success_std': len(vals.get('success_std', [])),
        'points_goodput': len(vals.get('goodput', [])),
        'points_throughput': len(vals.get('throughput', [])),
    })

pd.DataFrame(summary)

In [ ]:
label_map = {
    'LRFHSS': 'LR-FHSS',
    'LoRa_SF_rings': 'LoRa SF rings',
    'LRFHSS_3relays': 'LR-FHSS + 3 relays',
    'LoRa_SF_rings_3relays': 'LoRa SF rings + 3 relays',
}

def aligned_xy(metric_values):
    y = np.asarray(metric_values, dtype=float)
    n = min(len(n_nodes), len(y))
    return n_nodes[:n], y[:n]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

for scenario_name, vals in scenarios.items():
    display_name = label_map.get(scenario_name, scenario_name)

    x_s, y_s = aligned_xy(vals.get('success', []))
    x_std, y_std = aligned_xy(vals.get('success_std', []))
    axes[0].plot(x_s, y_s, marker='o', label=display_name)

    # add +- 1 std band only when lengths align
    if len(x_s) == len(x_std) and len(y_s) == len(y_std):
      lower = np.clip(y_s - y_std, 0.0, 1.0)
      upper = np.clip(y_s + y_std, 0.0, 1.0)
      axes[0].fill_between(x_s, lower, upper, alpha=0.15)

    x_g, y_g = aligned_xy(vals.get('goodput', []))
    axes[1].plot(x_g, y_g, marker='o', label=display_name)

    x_t, y_t = aligned_xy(vals.get('throughput', []))
    axes[2].plot(x_t, y_t, marker='o', label=display_name)

axes[0].set_title('Success Ratio vs Number of Nodes')
axes[0].set_xlabel('Number of Nodes')
axes[0].set_ylabel('Success Ratio')
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Goodput vs Number of Nodes')
axes[1].set_xlabel('Number of Nodes')
axes[1].set_ylabel('Goodput')
axes[1].grid(True, alpha=0.3)

axes[2].set_title('Throughput vs Number of Nodes')
axes[2].set_xlabel('Number of Nodes')
axes[2].set_ylabel('Throughput')
axes[2].grid(True, alpha=0.3)

handles, labels = axes[2].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.08))

plt.show()

In [ ]:
# Optional: tidy table for export/reporting
rows = []
for scenario_name, vals in scenarios.items():
    label = label_map.get(scenario_name, scenario_name)
    x_s, y_s = aligned_xy(vals.get('success', []))
    _, y_g = aligned_xy(vals.get('goodput', []))
    _, y_t = aligned_xy(vals.get('throughput', []))

    n = min(len(x_s), len(y_s), len(y_g), len(y_t))
    for i in range(n):
        rows.append({
            'nNodes': int(x_s[i]),
            'scenario': label,
            'success': float(y_s[i]),
            'goodput': float(y_g[i]),
            'throughput': float(y_t[i]),
        })

df = pd.DataFrame(rows).sort_values(['nNodes', 'scenario']).reset_index(drop=True)
df.head(20)